# FIRMS + Open-Meteo historical weather: weather-bearing candidate dataset

This is the direct training-data path: resume or collect the FIRMS/FEDS/terrain evidence, build the 1 km candidate rows, map each candidate cell to a compact weather location, retrieve ECMWF IFS hourly weather for that location and date, and join the hour at or before the row's prediction anchor. The result is a separate, uploadable weather-bearing dataset; the base candidate view stays immutable.

The weather anchor is the model prediction time (`anchor_at`), not a later time in the 12-hour label window. FIRMS acquisition timestamps remain the evidence used to seed and gate the candidate rows.

## Run order

1. Enable `WILDFIRE_RUN_NON_WEATHER_PIPELINE=1` to resume missing FIRMS/FEDS windows, normalize the required FEDS capture segments, collect missing terrain blocks, and build the positive and candidate views. Terminal FIRMS windows are skipped by default so immutable source files are not duplicated.
2. Pass that one completed candidate manifest to the weather backfill below. It calls Open-Meteo Historical Weather API with `models=ecmwf_ifs`, retains raw responses and the candidate-to-weather-tile mapping, and keeps the existing 600-location-unit/minute limiter.
3. Only a complete backfill can be joined and exported. A paused run is resumed from its partial manifest, which reuses already complete dates rather than re-requesting them. This prevents a partial weather collection from becoming an apparently complete upload.

These are retrospective historical-weather features for offline training. A live model must obtain equivalent current conditions or separately collected forecast-vintage inputs at inference time.

In [ ]:
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
import os
import sys

REPOSITORY_ROOT = Path.cwd().resolve()
SOURCE_ROOT = REPOSITORY_ROOT / "src"
if not SOURCE_ROOT.is_dir():
    raise RuntimeError(f"Run this notebook from the repository root; missing {SOURCE_ROOT}")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from dotenv import load_dotenv
load_dotenv(REPOSITORY_ROOT / "config/.env")

from wildfire_data.candidate_dataset import build_and_store_firms_candidate_dataset
from wildfire_data.collect_firms import collect_firms_range
from wildfire_data.data_archive import CoverageStatus
from wildfire_data.etopo_terrain import collect_etopo_terrain, context_tile_ids_from_detections
from wildfire_data.feds_collection import collect_feds_perimeters, rebuild_feds_primarykey_normalization
from wildfire_data.feds_labels import build_and_store_feds_weak_labels
from wildfire_data.forecast_tile_planning import iter_normalized_firms_detections
from wildfire_data.open_meteo_historical import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_CONSECUTIVE_RATE_LIMITS,
    DEFAULT_MAX_TILE_DISTANCE_METRES,
    DEFAULT_RATE_LIMIT_COOLDOWN_SECONDS,
    DEFAULT_REQUESTS_PER_MINUTE,
    backfill_open_meteo_historical_weather,
)
from wildfire_data.storage_budget import load_storage_budget
from wildfire_data.training_dataset import (
    DEFAULT_FIRMS_PRODUCTS,
    build_and_store_feds_weak_positive_training_dataset,
)
from wildfire_data.weather_candidate_dataset import (
    build_weather_candidate_dataset,
    export_weather_candidate_dataset_release,
)

DATA_ROOT = REPOSITORY_ROOT / "data"
STORAGE_POLICY = load_storage_budget()
HISTORICAL_START = date(2026, 5, 11)
HISTORICAL_END = date(2026, 8, 22)
FEDS_SOURCE_END = HISTORICAL_END + timedelta(days=1)

def environment_flag(name: str) -> bool:
    return os.getenv(name, "").strip().lower() in {"1", "true", "yes", "on"}

RUN_NON_WEATHER_PIPELINE = environment_flag("WILDFIRE_RUN_NON_WEATHER_PIPELINE")
REFRESH_TERMINAL_FIRMS = environment_flag("WILDFIRE_REFRESH_TERMINAL_FIRMS")
print(f"Archive root: {DATA_ROOT.resolve()} | UTC now: {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%SZ}")

In [ ]:
# Build all non-weather inputs before calling Open-Meteo. FIRMS needs one leading
# day for the 24-hour feature lookback; FEDS needs one trailing day for the final label.
FIRMS_COLLECTION_START = date(2026, 5, 10)
FIRMS_COLLECTION_END = HISTORICAL_END
FIRMS_PRODUCTS = DEFAULT_FIRMS_PRODUCTS
# These segments preserve one selected FEDS raw capture per non-overlapping source
# span in the existing archive. The middle 2026-05-31..2026-08-10 span is already
# normalized and labelled; re-running it would select a different capture and create
# duplicate immutable label artifacts.
FEDS_REBUILD_SEGMENTS = (
    (HISTORICAL_START, date(2026, 5, 30)),
    (date(2026, 8, 11), FEDS_SOURCE_END),
)
POSITIVE_VIEW_MANIFEST = None
configured_candidate_manifest = os.getenv("WILDFIRE_BASE_CANDIDATE_MANIFEST", "").strip()
BASE_CANDIDATE_MANIFEST = Path(configured_candidate_manifest) if configured_candidate_manifest else None

if RUN_NON_WEATHER_PIPELINE:
    api_key = (os.getenv("NASA_FIRMS_API_KEY") or os.getenv("MAP_KEY") or "").strip()
    if not api_key:
        raise RuntimeError("Set NASA_FIRMS_API_KEY (or MAP_KEY) before collecting FIRMS.")
    firms_result = collect_firms_range(
        str(DATA_ROOT),
        api_key=api_key,
        start_date=FIRMS_COLLECTION_START,
        end_date=FIRMS_COLLECTION_END,
        products=FIRMS_PRODUCTS,
        storage_budget=STORAGE_POLICY,
        refresh=REFRESH_TERMINAL_FIRMS,
    )
    print(f"FIRMS: {len(firms_result.responses):,} responses, {firms_result.skipped_terminal_count:,} terminal windows reused, {firms_result.failed_count:,} retries needed.")
    if firms_result.failed_count:
        raise RuntimeError("FIRMS collection is incomplete; rerun after resolving the recorded coverage failures.")

    feds_result = collect_feds_perimeters(
        str(DATA_ROOT),
        start_date=HISTORICAL_START,
        end_date=FEDS_SOURCE_END,
        storage_budget=STORAGE_POLICY,
    )
    print(f"FEDS source coverage: {feds_result.coverage.status.value}; {len(feds_result.windows):,} snapshot windows considered.")
    if feds_result.coverage.status not in {CoverageStatus.COMPLETE, CoverageStatus.EMPTY_CONFIRMED}:
        raise RuntimeError("FEDS source collection is incomplete; do not build labels from a partial capture.")

    for segment_start, segment_end in FEDS_REBUILD_SEGMENTS:
        normalization = rebuild_feds_primarykey_normalization(
            DATA_ROOT,
            storage_budget=STORAGE_POLICY,
            start_date=segment_start,
            end_date=segment_end,
        )
        print(f"FEDS normalization {segment_start}..{segment_end}: {normalization.status.value}, {normalization.feature_count:,} features.")
        if normalization.status is not CoverageStatus.COMPLETE:
            raise RuntimeError("FEDS normalization found incomplete or conflicting snapshots; resolve before labels.")

    label_reports = build_and_store_feds_weak_labels(
        DATA_ROOT,
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
        storage_budget=STORAGE_POLICY,
    )
    incomplete_labels = [report for report in label_reports if report.status not in {CoverageStatus.COMPLETE, CoverageStatus.EMPTY_CONFIRMED}]
    print(f"FEDS labels: {sum(report.positive_cell_count for report in label_reports):,} positives across {len(label_reports):,} source windows.")
    if incomplete_labels:
        raise RuntimeError(f"FEDS labels are incomplete for {len(incomplete_labels):,} source windows.")

    terrain_tile_ids = context_tile_ids_from_detections(
        iter_normalized_firms_detections(
            DATA_ROOT, start_date=FIRMS_COLLECTION_START, end_date=FIRMS_COLLECTION_END
        )
    )
    terrain_result = collect_etopo_terrain(
        DATA_ROOT,
        context_tile_ids=terrain_tile_ids,
        storage_budget=STORAGE_POLICY,
    )
    print(f"Terrain: {terrain_result.complete_count:,} complete, {terrain_result.skipped_count:,} reused, {terrain_result.partial_or_failed_count:,} incomplete blocks.")
    if terrain_result.partial_or_failed_count:
        raise RuntimeError("Terrain collection is incomplete; do not build a mixed candidate view.")

    positive_result = build_and_store_feds_weak_positive_training_dataset(
        DATA_ROOT,
        storage_budget=STORAGE_POLICY,
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
        firms_products=FIRMS_PRODUCTS,
    )
    POSITIVE_VIEW_MANIFEST = positive_result.manifest_path
    if POSITIVE_VIEW_MANIFEST is None:
        raise RuntimeError("No positive-only training rows were built for the requested source range.")
    print(f"Positive view: {positive_result.training_row_count:,} rows: {POSITIVE_VIEW_MANIFEST}")

    base_result = build_and_store_firms_candidate_dataset(
        DATA_ROOT,
        storage_budget=STORAGE_POLICY,
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
        split_start_date=HISTORICAL_START,
        split_end_date=HISTORICAL_END,
        positive_view_manifest=POSITIVE_VIEW_MANIFEST,
        firms_products=FIRMS_PRODUCTS,
    )
    BASE_CANDIDATE_MANIFEST = base_result.manifest_path
    print(f"Base candidates: {base_result.candidate_row_count:,} rows: {BASE_CANDIDATE_MANIFEST}")
else:
    print("Non-weather collection is disabled. Set WILDFIRE_RUN_NON_WEATHER_PIPELINE=1 to collect or resume source evidence before weather.")

In [ ]:
# Weather is intentionally in the next cell. This keeps every non-weather source
# stage complete before the rate-limited provider is contacted.
if BASE_CANDIDATE_MANIFEST is not None:
    print(f"Candidate spine selected: {BASE_CANDIDATE_MANIFEST}")
else:
    print("No candidate spine selected yet. Run the non-weather pipeline or set WILDFIRE_BASE_CANDIDATE_MANIFEST.")

In [ ]:
# Historical Open-Meteo weather: candidate location cover at its anchor hour.
# The default 10 km cover reduces calls; its mapping records the actual distance. Lower it for less spatial coalescing.
# The collector batches coordinates but charges every location to the rate limiter.
OPEN_METEO_MODEL = "ecmwf_ifs"
OPEN_METEO_REQUESTS_PER_MINUTE = DEFAULT_REQUESTS_PER_MINUTE  # Keep 600 unless the provider directs otherwise.
OPEN_METEO_BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_TILE_DISTANCE_M = DEFAULT_MAX_TILE_DISTANCE_METRES
RATE_LIMIT_COOLDOWN_SECONDS = DEFAULT_RATE_LIMIT_COOLDOWN_SECONDS
MAX_CONSECUTIVE_429S = DEFAULT_MAX_CONSECUTIVE_RATE_LIMITS
configured_resume_manifest = os.getenv("WILDFIRE_RESUME_WEATHER_BACKFILL_MANIFEST", "").strip()
RESUME_WEATHER_BACKFILL_MANIFEST = Path(configured_resume_manifest) if configured_resume_manifest else None
BACKFILL_HISTORICAL_WEATHER = environment_flag("WILDFIRE_BACKFILL_HISTORICAL_WEATHER")
configured_weather_backfill = os.getenv("WILDFIRE_WEATHER_BACKFILL_MANIFEST", "").strip()
WEATHER_BACKFILL_MANIFEST = Path(configured_weather_backfill) if configured_weather_backfill else None

if BACKFILL_HISTORICAL_WEATHER:
    if BASE_CANDIDATE_MANIFEST is None:
        raise RuntimeError("Set BASE_CANDIDATE_MANIFEST; weather must be tied to one completed candidate view.")
    backfill_result = backfill_open_meteo_historical_weather(
        DATA_ROOT,
        storage_policy=STORAGE_POLICY,
        candidate_manifest=BASE_CANDIDATE_MANIFEST,
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
        resume_manifest=RESUME_WEATHER_BACKFILL_MANIFEST,
        model=OPEN_METEO_MODEL,
        max_tile_distance_m=MAX_TILE_DISTANCE_M,
        requests_per_minute=OPEN_METEO_REQUESTS_PER_MINUTE,
        batch_size=OPEN_METEO_BATCH_SIZE,
        rate_limit_cooldown_seconds=RATE_LIMIT_COOLDOWN_SECONDS,
        max_consecutive_rate_limits=MAX_CONSECUTIVE_429S,
    )
    WEATHER_BACKFILL_MANIFEST = backfill_result.manifest_path
    print(f"Weather backfill complete={backfill_result.complete}; manifest: {WEATHER_BACKFILL_MANIFEST}")
    if not backfill_result.complete:
        raise RuntimeError("Weather backfill paused or failed. Do not join it; set RESUME_WEATHER_BACKFILL_MANIFEST to this manifest and rerun the cell.")
else:
    print("Historical weather backfill is disabled. The 600-location-unit/minute limiter remains the default.")

In [ ]:
# Join only a complete backfill, then create a portable upload directory.
BUILD_WEATHER_DATASET = environment_flag("WILDFIRE_BUILD_WEATHER_DATASET")
configured_weather_candidate = os.getenv("WILDFIRE_WEATHER_CANDIDATE_MANIFEST", "").strip()
WEATHER_CANDIDATE_MANIFEST = Path(configured_weather_candidate) if configured_weather_candidate else None
EXPORT_WEATHER_RELEASE = environment_flag("WILDFIRE_EXPORT_WEATHER_RELEASE")
WEATHER_RELEASE_DIRECTORY = Path("releases/wildfire-spread-firms-feds-weather-2026-05-11_to_2026-08-22")

if BUILD_WEATHER_DATASET:
    if BASE_CANDIDATE_MANIFEST is None or WEATHER_BACKFILL_MANIFEST is None:
        raise RuntimeError("Set the matching base-candidate and complete weather-backfill manifests first.")
    weather_dataset_result = build_weather_candidate_dataset(
        DATA_ROOT,
        storage_budget=STORAGE_POLICY,
        candidate_manifest=BASE_CANDIDATE_MANIFEST,
        weather_backfill_manifest=WEATHER_BACKFILL_MANIFEST,
    )
    WEATHER_CANDIDATE_MANIFEST = weather_dataset_result.manifest_path
    print(f"Built {weather_dataset_result.candidate_row_count:,} weather-bearing rows: {WEATHER_CANDIDATE_MANIFEST}")

if EXPORT_WEATHER_RELEASE:
    if WEATHER_CANDIDATE_MANIFEST is None:
        raise RuntimeError("Set WEATHER_CANDIDATE_MANIFEST before export.")
    release = export_weather_candidate_dataset_release(
        DATA_ROOT,
        WEATHER_RELEASE_DIRECTORY,
        weather_candidate_manifest=WEATHER_CANDIDATE_MANIFEST,
    )
    print(f"Exported {release.candidate_row_count:,} weather-bearing rows to {release.directory}")

## Optional forward forecast capture

This is separate from the retrospective training backfill. Run it only while a forecast is operational, name the exact model run yourself, and keep its resulting issued-forecast features out of the historical-analysis upload.

In [ ]:
# Forward-only operational experiment: do not use this to recreate historical forecast availability.
# It reuses the 600 location-unit/minute rate limit and records successful-response availability.
import pandas as pd
from wildfire_data.forecast_tile_planning import iter_normalized_firms_detections
from wildfire_data.open_meteo_single_run import (
    DEFAULT_CANDIDATE_RADIUS_CELLS,
    DEFAULT_FORECAST_HORIZON_HOURS,
    capture_open_meteo_single_run,
    plan_firms_candidate_weather_tiles,
)

CAPTURE_FORWARD_FORECAST = False
FORWARD_FIRMS_START = datetime.now(timezone.utc).date()
FORWARD_FIRMS_END = FORWARD_FIRMS_START
FORWARD_MODEL = "ecmwf_ifs"
FORWARD_MODEL_RUN_AT = None  # Example: "2026-08-26T12:00:00Z" after that named run is accessible.

if CAPTURE_FORWARD_FORECAST:
    if FORWARD_MODEL_RUN_AT is None:
        raise RuntimeError("Set FORWARD_MODEL_RUN_AT to an explicit, already available UTC run.")
    forward_firms = pd.DataFrame(
        {
            "detection_id": record["detection_id"],
            "latitude": record["latitude"],
            "longitude": record["longitude"],
            "acquired_at": record["acquired_at"],
            "raw_artifact_id": record.get("provenance", {}).get("raw_artifact_id"),
        }
        for record in iter_normalized_firms_detections(
            DATA_ROOT, start_date=FORWARD_FIRMS_START, end_date=FORWARD_FIRMS_END
        )
    )
    if forward_firms.empty:
        raise RuntimeError("No normalized FIRMS detections exist for the forward window.")
    forward_plan = plan_firms_candidate_weather_tiles(
        forward_firms, candidate_radius_cells=DEFAULT_CANDIDATE_RADIUS_CELLS
    )
    forward_capture = capture_open_meteo_single_run(
        DATA_ROOT,
        forward_plan,
        model=FORWARD_MODEL,
        model_run_at=FORWARD_MODEL_RUN_AT,
        forecast_horizon_hours=DEFAULT_FORECAST_HORIZON_HOURS,
        storage_policy=STORAGE_POLICY,
        requests_per_minute=OPEN_METEO_REQUESTS_PER_MINUTE,
        batch_size=OPEN_METEO_BATCH_SIZE,
        rate_limit_cooldown_seconds=RATE_LIMIT_COOLDOWN_SECONDS,
        max_consecutive_rate_limits=MAX_CONSECUTIVE_429S,
    )
    print(f"Forward capture: {forward_capture.captured_tile_count:,}/{forward_capture.planned_tile_count:,} tiles; paused={forward_capture.paused_for_rate_limit}.")
else:
    print("Forward forecast capture is disabled.")

## What the upload contains

Each weather-bearing candidate row has `weather_temperature_2m`, `weather_relative_humidity_2m`, `weather_precipitation`, `weather_wind_u_10m`, and `weather_wind_v_10m`, plus the model, mapped weather tile, raw artifact IDs, and `weather_observed_at`. The join refuses a missing field, another row's tile, a later weather hour, or a partial backfill. ECMWF weather is coarser than the 1 km training grid; with the default cover a mapped request location may additionally be up to 10 km from a candidate centre, and the candidate-to-request distance plus returned grid location are retained.

For a live prediction service, collect the same features at the live prediction time. Open-Meteo Single Runs can remain a separate, forward-only forecast-vintage experiment; it must not be mixed with this retrospective training dataset.